# JEPA Mini Project Colab Pro Runner

This notebook is the GPU-oriented path for Colab Pro. It mounts Drive, installs dependencies, runs either a standard MAE-vs-JEPA comparison or a JEPA tuning sweep, and then launches downstream evaluation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

WORKSPACE_ROOT = Path('/content')
PROJECT_ROOT = WORKSPACE_ROOT / 'jepa_mini_project'
if PROJECT_ROOT.exists():
    os.chdir(PROJECT_ROOT)
print('Working directory:', Path.cwd())
print('If the repo is not here yet, upload it or clone it into /content/jepa_mini_project before continuing.')

In [ ]:
!python3 -m pip install -r requirements.txt

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## Option A: Standard GPU Compare

Run one stronger JEPA config and one MAE baseline.

In [ ]:
!python3 -m src.train_jepa --config configs/colab_pro_jepa.yaml
!python3 -m src.train_mae --config configs/colab_pro.yaml

## Option B: JEPA Sweep

If you want to test whether JEPA can close the gap, run the sweep script instead of Option A.

In [ ]:
# !bash scripts/run_colab_jepa_sweep.sh

In [ ]:
from pathlib import Path

run_dirs = sorted(Path('runs').glob('*'), key=lambda path: path.stat().st_mtime)
jepa_ckpt = None
mae_ckpt = None
for run_dir in reversed(run_dirs):
    if 'jepa' in run_dir.name and jepa_ckpt is None:
        jepa_ckpt = run_dir / 'checkpoints' / 'best.ckpt'
    if 'mae' in run_dir.name and mae_ckpt is None:
        mae_ckpt = run_dir / 'checkpoints' / 'best.ckpt'
print('JEPA checkpoint:', jepa_ckpt)
print('MAE checkpoint:', mae_ckpt)

In [ ]:
import subprocess

subprocess.run([
    'python3', '-m', 'src.eval_linear_probe', '--checkpoint', str(jepa_ckpt), '--config', 'configs/colab_pro_jepa.yaml'
], check=True)
subprocess.run([
    'python3', '-m', 'src.eval_retrieval', '--checkpoint', str(jepa_ckpt), '--config', 'configs/colab_pro_jepa.yaml'
], check=True)
subprocess.run([
    'python3', '-m', 'src.export_latents', '--checkpoint', str(jepa_ckpt), '--config', 'configs/colab_pro_jepa.yaml', '--split', 'test'
], check=True)

if mae_ckpt is not None:
    subprocess.run([
        'python3', '-m', 'src.eval_linear_probe', '--checkpoint', str(mae_ckpt), '--config', 'configs/colab_pro.yaml'
    ], check=True)
    subprocess.run([
        'python3', '-m', 'src.eval_retrieval', '--checkpoint', str(mae_ckpt), '--config', 'configs/colab_pro.yaml'
    ], check=True)
    subprocess.run([
        'python3', '-m', 'src.export_latents', '--checkpoint', str(mae_ckpt), '--config', 'configs/colab_pro.yaml', '--split', 'test'
    ], check=True)
    subprocess.run([
        'python3', '-m', 'src.visualize_embeddings', '--checkpoint', str(jepa_ckpt), '--checkpoint', str(mae_ckpt), '--config', 'configs/colab_pro.yaml'
    ], check=True)

In [ ]:
from IPython.display import Image, display
from pathlib import Path

viz_candidates = sorted(Path('runs').glob('*/embedding_viz/embedding_comparison.png'))
if viz_candidates:
    display(Image(filename=str(viz_candidates[-1])))
else:
    print('No embedding plot found yet.')